In [9]:
import numpy as np
from datetime import datetime, timedelta
import string

def parse_chat(file_path):
    messages = []
    sys_msgs, media_omitted, deleted_msgs = 0, 0, 0

    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        if not line:
            continue

        if len(line) > 8 and line[2] == '/' and line[5] == '/':
            parts = line.split(' - ', 1)
            if len(parts) < 2:
                continue

            timestamp_str = parts[0].strip()
            rest = parts[1]

            if ':' not in rest:
                sys_msgs += 1
                continue

            sender_parts = rest.split(':', 1)
            sender = sender_parts[0].strip()
            text = sender_parts[1].strip()

            if text == '<Media omitted>':
                media_omitted += 1
                is_valid = False
            elif text == 'This message was deleted':
                deleted_msgs += 1
                is_valid = False
            else:
                is_valid = True

            dt = datetime.strptime(timestamp_str, '%d/%m/%y, %H:%M')

            messages.append({
                'timestamp': dt,
                'sender': sender,
                'text': text,
                'is_valid': is_valid
            })

    print(f"Successfully parsed {len(messages)} messages, skipped {sys_msgs} system messages, {media_omitted} media-omitted, {deleted_msgs} deleted messages.")
    return messages

def get_group_overview(messages):
    valid_msgs = [m for m in messages if m['is_valid']]
    total_messages = len(valid_msgs)

    if total_messages == 0:
        return None

    start_date = valid_msgs[0]['timestamp'].date()
    end_date = valid_msgs[-1]['timestamp'].date()
    total_days = (end_date - start_date).days + 1

    senders = {}
    for m in valid_msgs:
        senders[m['sender']] = senders.get(m['sender'], 0) + 1

    sorted_senders = sorted(senders.items(), key=lambda x: x[1], reverse=True)

    return {
        'start': start_date, 'end': end_date, 'days': total_days,
        'total': total_messages, 'participants': len(senders),
        'leaderboard': sorted_senders
    }

def get_activity_peaks(messages):
    valid_msgs = [m for m in messages if m['is_valid']]
    day_counts = {}
    hour_counts = {}

    for m in valid_msgs:
        day_str = m['timestamp'].strftime('%d %B %Y')
        hour = m['timestamp'].hour
        day_counts[day_str] = day_counts.get(day_str, 0) + 1
        hour_counts[hour] = hour_counts.get(hour, 0) + 1

    busiest_day = max(day_counts.items(), key=lambda x: x[1])
    busiest_hour = max(hour_counts.items(), key=lambda x: x[1])

    return busiest_day, busiest_hour

def generate_heatmap(messages, participants):
    valid_msgs = [m for m in messages if m['is_valid']]
    matrix = np.zeros((len(participants), 24), dtype=int)

    for m in valid_msgs:
        p_idx = participants.index(m['sender'])
        h_idx = m['timestamp'].hour
        matrix[p_idx, h_idx] += 1

    return matrix

def render_heatmap(matrix, participants):
    print("ACTIVITY HEATMAP (messages by hour)")
    print("      00 03 06 09 12 15 18 21")
    for i, p in enumerate(participants):
        row = matrix[i]
        max_val = max(row) if max(row) > 0 else 1
        row_str = ""
        for h in range(24):
            pct = (row[h] / max_val) * 100
            if pct == 0: char = " "
            elif pct <= 25: char = "."
            elif pct <= 50: char = "-"
            elif pct <= 75: char = "="
            else: char = "#"
            row_str += char
        print(f"{p:<6} {row_str}")

def get_top_words(messages):
    valid_msgs = [m for m in messages if m['is_valid']]
    stop_words = {'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'it', 'my', 'that', 'with', 'you', 'this', 'have', 'be', 'so'}
    word_counts = {}

    for m in valid_msgs:
        clean_text = m['text'].translate(str.maketrans('', '', string.punctuation)).lower()
        words = clean_text.split()
        for w in words:
            if w not in stop_words and len(w) > 2:
                word_counts[w] = word_counts.get(w, 0) + 1

    return sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:10]

def get_response_patterns(messages, participants, total_days, start_date):
    valid_msgs = [m for m in messages if m['is_valid']]

    response_times = {p: [] for p in participants}
    last_sender = None
    last_time = None

    for m in valid_msgs:
        if last_sender and last_sender != m['sender']:
            gap = (m['timestamp'] - last_time).total_seconds()
            if gap < 3600 * 24:
                response_times[m['sender']].append(gap)
        last_sender = m['sender']
        last_time = m['timestamp']

    avg_responses = {}
    for p, gaps in response_times.items():
        if gaps:
            avg_responses[p] = sum(gaps) / len(gaps)
        else:
            avg_responses[p] = float('inf')

    streaks = {p: 0 for p in participants}
    for p in participants:
        p_dates = {m['timestamp'].date() for m in valid_msgs if m['sender'] == p}
        max_streak = 0
        current_streak = 0
        for i in range(total_days):
            d = start_date + timedelta(days=i)
            if d not in p_dates:
                current_streak += 1
                max_streak = max(max_streak, current_streak)
            else:
                current_streak = 0
        streaks[p] = max_streak

    return avg_responses, streaks

def detect_archetypes(messages, participants, heatmap_matrix, overview):
    archetypes = {p: [] for p in participants}
    caring_keywords = {'okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please', 'reminder', 'drink water', 'dont forget'}

    valid_msgs = [m for m in messages if m['is_valid']]

    for i, p in enumerate(participants):
        p_msgs = [m for m in valid_msgs if m['sender'] == p]
        if not p_msgs:
            archetypes[p] = ("THE GHOST", "(0 messages)")
            continue

        total_p = len(p_msgs)

        bursts = []
        current_burst = 0
        last_sender = None
        for m in valid_msgs:
            if m['sender'] == p:
                if last_sender == p:
                    current_burst += 1
                else:
                    if current_burst > 0: bursts.append(current_burst)
                    current_burst = 1
            last_sender = m['sender']
        avg_burst = sum(bursts)/len(bursts) if bursts else 0

        mom_score = sum(1 for m in p_msgs if any(k in m['text'].lower() for k in caring_keywords))

        night_msgs = np.sum(heatmap_matrix[i, [23, 0, 1, 2, 3, 4]])
        owl_pct = (night_msgs / total_p) * 100 if total_p > 0 else 0

        avg_words = sum(len(m['text'].split()) for m in p_msgs) / total_p

        drama_count = sum(1 for m in p_msgs if (m['text'].isupper() and len(m['text']) > 2) or m['text'].count('!') >= 2)
        drama_pct = (drama_count / total_p) * 100

        p_dates = len(set(m['timestamp'].date() for m in p_msgs))
        ghost_pct = ((overview['days'] - p_dates) / overview['days']) * 100

        if ghost_pct > 60:
            archetypes[p] = ("THE GHOST", f"(silent {int(ghost_pct)}% of days)")
        elif owl_pct > 60:
            archetypes[p] = ("THE NIGHT OWL", f"({owl_pct:.1f}% msgs after 11 PM)")
        elif drama_pct > 30:
            archetypes[p] = ("THE DRAMA QUEEN", f"({drama_pct:.1f}% ALL-CAPS msgs)")
        elif avg_words > 30:
            archetypes[p] = ("THE STORYTELLER", f"(avg {avg_words:.1f} words per msg)")
        elif avg_burst > 3:
            archetypes[p] = ("THE SPAMMER", f"(avg {avg_burst:.1f} msgs in a row)")
        elif mom_score > (total_p * 0.1):
            archetypes[p] = ("THE GROUP MOM", f"({mom_score} caring keywords)")
        else:
            q_pct = sum(1 for m in p_msgs if m['text'].endswith('?')) / total_p * 100
            archetypes[p] = ("THE QUESTION MASTER", f"({q_pct:.1f}% questions)")

    return archetypes

def print_report(overview, busiest, heatmap, matrix, participants, top_words, responses, streaks, archetypes):
    print("\n" + "="*60)
    print("GROUPDNA REPORT".center(60))
    print("="*60)

    print(f"Group: 'Hostel Bois 4ever'")
    print(f"Period: {overview['start'].strftime('%d %B %Y')} to {overview['end'].strftime('%d %B %Y')} ({overview['days']} days)")
    print(f"Total messages: {overview['total']}")
    print(f"Participants: {overview['participants']}")
    print("="*60)

    print(f"Busiest day  : {busiest[0][0]} ({busiest[0][1]} messages)")
    print(f"Busiest hour : {busiest[1][0]:02d}:00 - {busiest[1][0]+1:02d}:00 (avg {busiest[1][1]//overview['days']} messages per day)\n")

    print("MESSAGES PER PERSON")
    for p, count in overview['leaderboard']:
        pct = (count / overview['total']) * 100
        print(f"{p:<10} {count} ({pct:.1f}%)")

    print("\n")
    render_heatmap(matrix, participants)
    print("\n")

    print("THIS GROUP'S FAVOURITE WORDS")
    for w, count in top_words:
        bar = "█" * (count // (top_words[0][1] // 15 + 1))
        print(f"{w:<10} {count:<5} {bar}")

    print("\nRESPONSE PATTERNS")
    fastest = min(responses, key=responses.get)
    slowest = max(responses, key=responses.get)
    print(f"Fastest replier : {fastest} (avg {responses[fastest]/60:.1f} minutes)")
    print(f"Slowest replier : {slowest} (avg {responses[slowest]/3600:.1f} hours)\n")

    print("LONGEST SILENT STREAKS")
    sorted_streaks = sorted(streaks.items(), key=lambda x: x[1], reverse=True)
    for p, days in sorted_streaks:
        print(f"{p:<10} : {days} days")

    print("\nPERSONALITY ARCHETYPES")
    for p, (arch, reason) in archetypes.items():
        print(f"{p:<10} -> {arch:<20} {reason}")

    print("\n" + "="*60)
    print("Generated by GroupDNA".center(60))
    print("Built with Python + NumPy".center(60))
    print("="*60)

if __name__ == "__main__":
    messages = parse_chat('hostel_bois.txt')
    overview = get_group_overview(messages)
    busiest = get_activity_peaks(messages)

    participants = [p for p, _ in overview['leaderboard']]
    heatmap_matrix = generate_heatmap(messages, participants)
    top_words = get_top_words(messages)

    responses, streaks = get_response_patterns(messages, participants, overview['days'], overview['start'])
    archetypes = detect_archetypes(messages, participants, heatmap_matrix, overview)

    print_report(overview, busiest, heatmap_matrix, heatmap_matrix, participants, top_words, responses, streaks, archetypes)

FileNotFoundError: [Errno 2] No such file or directory: 'hostel_bois.txt'